In [ ]:
import numpy as np
import pandas as pd
import joblib
import torch
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, f1_score, confusion_matrix
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)

In [ ]:
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)


In [ ]:
TRAIN_FILE = "TrainingDataset.csv"
VAL_FILE = "ValidationDataset.csv"
TEST_FILE = "TestingDataset.csv"
TEXT_COL = 0       # preprocessed text column
LABEL_COL = 1
MODEL_CHECKPOINT = "UBC-NLP/MARBERT"   # pretrained on ~1B Arabic (mostly dialectal) tweets
MAX_LENGTH = 64          # tweets are short; 64 tokens comfortably covers most
FINAL_EPOCHS = 4         # epochs for the winning config's final training run
SEARCH_EPOCHS = 3        # shorter budget per config during the grid search

# Note: MARBERT was pretrained directly on ~1B Arabic tweets (mostly dialectal,
# not MSA), including the informal spelling, emojis, and dialect variation
# typical of tweets. Unlike AraBERT, it does not expect farasa-segmented input,
# so no additional model-specific preprocessing step is needed here beyond the
# generic cleaning already done in your other script. This is one reason
# MARBERT is often a strong fit for tweet-style classification tasks
# specifically, as opposed to AraBERT's MSA/news-heavy pretraining corpus.

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}\n")

Using device: cuda



In [ ]:
# ==========================================
# 1. LOAD DATA (already preprocessed, deduplicated, and split)
# ==========================================
train_df = pd.read_csv(TRAIN_FILE, header=None)
val_df = pd.read_csv(VAL_FILE, header=None)
test_df = pd.read_csv(TEST_FILE, header=None)

print("Dataset Sizes Loaded:")
print(f" - Train Set:      {len(train_df)} samples")
print(f" - Validation Set: {len(val_df)} samples")
print(f" - Test Set:       {len(test_df)} samples\n")

Dataset Sizes Loaded:
 - Train Set:      108786 samples
 - Validation Set: 14510 samples
 - Test Set:       21757 samples



In [ ]:
train_df

,0,1
0,تشيلسي يتعادل مع مانشستر يونايتد بهدف لمثله في...,رياضه
1,فوائد لا تعرفينها من قبل عن فاكهه الدراجون,طهي
2,يدوم جمال المرأه مادام الرجل يحبها ويدوم حب ال...,خواطر واقتباسات
3,خادم الحرمين الشريفين الملك سلمان بن عبدالعزيز...,سياسه
4,بورشه تيربو الكلاسيكيه من قطع الليجو,سيارت
...,...,...
108781,وصفها ترمب بالأخبار العظيمه شركه موديرنا تعلن ...,صحه
108782,اللعبه ستكون مبنيه علي محرك ولكنها ستكون مختلف...,تكنولوجيا
108783,بطش أردوغان يتفاقم والأتراك يهربون للخارج تركيا,سياسه
108784,بلتون الماليه تحدد السهم الاستهلاكي الأكثر تأث...,اقتصاد


In [ ]:
# ==========================================
# 2. LABEL ENCODING
# ==========================================
label_encoder = LabelEncoder()
train_df["label"] = label_encoder.fit_transform(train_df[LABEL_COL])
val_df["label"] = label_encoder.transform(val_df[LABEL_COL])
test_df["label"] = label_encoder.transform(test_df[LABEL_COL])

num_classes = len(label_encoder.classes_)
print(f"Classes ({num_classes}): {list(label_encoder.classes_)}\n")

Classes (10): ['اقتصاد', 'تكنولوجيا', 'خواطر واقتباسات', 'دين', 'رياضه', 'سيارت', 'سياسه', 'صحه', 'طهي', 'علوم']



In [ ]:
# ==========================================
# 3. TOKENIZATION (HuggingFace Datasets + MARBERT tokenizer)
# ==========================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

def tokenize_batch(batch):
    return tokenizer(
       batch[str(TEXT_COL)], # Access using string key '0'
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
    )

train_ds = Dataset.from_pandas(train_df[[TEXT_COL, "label"]])
val_ds = Dataset.from_pandas(val_df[[TEXT_COL, "label"]])
test_ds = Dataset.from_pandas(test_df[[TEXT_COL, "label"]])

train_ds = train_ds.map(tokenize_batch, batched=True)
val_ds = val_ds.map(tokenize_batch, batched=True)
test_ds = test_ds.map(tokenize_batch, batched=True)

keep_cols = ["input_ids", "attention_mask", "label"]
train_ds.set_format(type="torch", columns=keep_cols)
val_ds.set_format(type="torch", columns=keep_cols)
test_ds.set_format(type="torch", columns=keep_cols)

config.json:   0%|          | 0.00/701 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/376 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/1.10M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

/usr/local/lib/python3.13/dist-packages/datasets/table.py:728: UserWarning: The DataFrame has column names of mixed type. They will be converted to strings and not roundtrip correctly.
  return cls(pa.Table.from_pandas(*args, **kwargs))
/usr/local/lib/python3.13/dist-packages/datasets/table.py:728: UserWarning: The DataFrame has column names of mixed type. They will be converted to strings and not roundtrip correctly.
  return cls(pa.Table.from_pandas(*args, **kwargs))
/usr/local/lib/python3.13/dist-packages/datasets/table.py:728: UserWarning: The DataFrame has column names of mixed type. They will be converted to strings and not roundtrip correctly.
  return cls(pa.Table.from_pandas(*args, **kwargs))


Map:   0%|          | 0/108786 [00:00<?, ? examples/s]

Map:   0%|          | 0/14510 [00:00<?, ? examples/s]

Map:   0%|          | 0/21757 [00:00<?, ? examples/s]

In [ ]:
# ==========================================
# 4. METRICS
# ==========================================
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "f1_macro": f1_score(labels, preds, average="macro"),
        "f1_weighted": f1_score(labels, preds, average="weighted"),
    }

In [ ]:
# ==========================================
# 5. HYPERPARAMETER TUNING (grid search, selected on validation macro-F1)
# ==========================================
# Same grid and selection discipline as the AraBERT script, so the two
# transformer baselines are tuned and compared on equal footing.
LEARNING_RATE_GRID = [2e-5, 5e-5]
BATCH_SIZE_GRID = [16, 32]

# Disable torchvision integration in datasets to prevent VideoReader import error
import datasets
datasets.config.TORCHVISION_AVAILABLE = False

best_val_f1_macro = -1.0
best_config = None

print("Starting hyperparameter grid search (selecting on validation macro-F1)...\n")
for lr in LEARNING_RATE_GRID:
    for batch_size in BATCH_SIZE_GRID:
        run_name = f"lr{lr}_bs{batch_size}"
        print(f"--- Config: learning_rate={lr}, batch_size={batch_size} ---")

        model = AutoModelForSequenceClassification.from_pretrained(
            MODEL_CHECKPOINT, num_labels=num_classes
        ).to(DEVICE)

        training_args = TrainingArguments(
            output_dir=f"./marbert_search_{run_name}",
            num_train_epochs=SEARCH_EPOCHS,
            per_device_train_batch_size=batch_size,
            per_device_eval_batch_size=batch_size,
            learning_rate=lr,
            eval_strategy="epoch",
            save_strategy="epoch",
            save_total_limit=1,
            load_best_model_at_end=True,
            metric_for_best_model="f1_macro",
            greater_is_better=True,
            logging_steps=50,
            report_to="none",
            seed=SEED,
        )

        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=train_ds,
            eval_dataset=val_ds,
            compute_metrics=compute_metrics,
            callbacks=[EarlyStoppingCallback(early_stopping_patience=1)],
        )

        trainer.train()
        val_metrics = trainer.evaluate()
        val_f1_macro = val_metrics["eval_f1_macro"]
        print(f"    Validation Macro-F1: {val_f1_macro:.4f}\n")

        if val_f1_macro > best_val_f1_macro:
            best_val_f1_macro = val_f1_macro
            best_config = {"learning_rate": lr, "batch_size": batch_size}

print("Best Hyperparameters Found:")
print(best_config)
print(f"Best Validation Macro-F1 (search budget): {best_val_f1_macro:.4f}\n")


Starting hyperparameter grid search (selecting on validation macro-F1)...

--- Config: learning_rate=2e-05, batch_size=16 ---


pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  654MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: UBC-NLP/MARBERT
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider

model.safetensors: reconstructing file:   0%|          |  0.00B /  654MB            

... (27 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,F1 Macro,F1 Weighted
0.053547,0.342928,3,0.931325,0.932244


    Validation Macro-F1: 0.9313

Best Hyperparameters Found:
{'learning_rate': 2e-05, 'batch_size': 32}
Best Validation Macro-F1 (search budget): 0.9403



In [ ]:
# ==========================================
# 6. FINAL TRAINING RUN (winning config, fuller epoch budget)
# ==========================================
final_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CHECKPOINT, num_labels=num_classes
).to(DEVICE)

final_training_args = TrainingArguments(
    output_dir="./marbert_final",
    num_train_epochs=FINAL_EPOCHS,
    per_device_train_batch_size=best_config["batch_size"],
    per_device_eval_batch_size=best_config["batch_size"],
    learning_rate=best_config["learning_rate"],
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    logging_steps=50,
    report_to="none",
    seed=SEED,
)

final_trainer = Trainer(
    model=final_model,
    args=final_training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

print("Training final model with best hyperparameters...")
final_trainer.train()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: UBC-NLP/MARBERT
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider

Training final model with best hyperparameters...


Epoch,Training Loss,Validation Loss,F1 Macro,F1 Weighted
1,0.250112,0.223827,0.926802,0.927924
2,0.162153,0.221168,0.934816,0.935717
3,0.059157,0.301660,0.936657,0.937498
4,0.031474,0.344161,0.939045,0.939905


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=13600, training_loss=0.14000534139573573, metrics={'train_runtime': 4552.8028, 'train_samples_per_second': 95.577, 'train_steps_per_second': 2.987, 'total_flos': 1.4312427604457472e+16, 'train_loss': 0.14000534139573573, 'epoch': 4.0})

In [ ]:
# ==========================================
# 7. VALIDATION SET EVALUATION (final chosen model)
# ==========================================
val_metrics = final_trainer.evaluate()
print("==========================================")
print("VALIDATION SET RESULTS (final model)")
print("==========================================")
print(f"Validation Macro-F1:    {val_metrics['eval_f1_macro']:.4f}")
print(f"Validation Weighted-F1: {val_metrics['eval_f1_weighted']:.4f}\n")

# ==========================================
# 8. FINAL TEST SET EVALUATION (once, on held-out test set)
# ==========================================
test_predictions = final_trainer.predict(test_ds)
y_test_pred_int = np.argmax(test_predictions.predictions, axis=1)
y_test_int = test_df["label"].values

test_f1_macro = f1_score(y_test_int, y_test_pred_int, average="macro")
test_f1_weighted = f1_score(y_test_int, y_test_pred_int, average="weighted")

print("==========================================")
print("FINAL UNSEEN TEST SET RESULTS")
print("==========================================")
print(f"Test Macro-F1:    {test_f1_macro:.4f}")
print(f"Test Weighted-F1: {test_f1_weighted:.4f}\n")

print("Full Classification Report:")
print(classification_report(
    y_test_int, y_test_pred_int, target_names=label_encoder.classes_
))

print("Confusion Matrix (rows=true, cols=predicted):")
cm = confusion_matrix(y_test_int, y_test_pred_int)
cm_df = pd.DataFrame(cm, index=label_encoder.classes_, columns=label_encoder.classes_)
print(cm_df, "\n")

# ==========================================
# 9. SAVING MODEL, TOKENIZER, LABEL ENCODER
# ==========================================
final_trainer.save_model("marbert_final_model")
tokenizer.save_pretrained("marbert_final_model")
joblib.dump(label_encoder, "marbert_label_encoder.joblib")
joblib.dump(best_config, "marbert_best_config.joblib")
print("Saved model, tokenizer, label encoder, and best config to disk.")

Training Loss,Validation Loss,Epoch,F1 Macro,F1 Weighted
0.031474,0.344161,4,0.939045,0.939905


VALIDATION SET RESULTS (final model)
Validation Macro-F1:    0.9390
Validation Weighted-F1: 0.9399



FINAL UNSEEN TEST SET RESULTS
Test Macro-F1:    0.9402
Test Weighted-F1: 0.9411

Full Classification Report:
                 precision    recall  f1-score   support

         اقتصاد       0.93      0.95      0.94      2239
      تكنولوجيا       0.95      0.94      0.94      2328
خواطر واقتباسات       0.94      0.94      0.94      2447
            دين       0.94      0.93      0.93      1957
          رياضه       0.99      0.99      0.99      2327
          سيارت       0.97      0.97      0.97      2214
          سياسه       0.96      0.95      0.95      2437
            صحه       0.89      0.91      0.90      2099
            طهي       0.97      0.96      0.96      1719
           علوم       0.88      0.87      0.87      1990

       accuracy                           0.94     21757
      macro avg       0.94      0.94      0.94     21757
   weighted avg       0.94      0.94      0.94     21757

Confusion Matrix (rows=true, cols=predicted):
                 اقتصاد  تكنولوجيا  خواطر وا

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved model, tokenizer, label encoder, and best config to disk.


In [ ]:
predicted_class_names = label_encoder.inverse_transform(y_test_pred_int)

# Create a DataFrame to display test tweets with their predicted classes
results_df = pd.DataFrame({
    'Tweet': test_df[TEXT_COL].values,
    'Actual_Class': label_encoder.inverse_transform(y_test_int),
    'Predicted_Class': predicted_class_names
})

print("Test Tweets with Predicted Classes (First 10 samples):")
display(results_df.head(10))

Test Tweets with Predicted Classes (First 10 samples):


,Tweet,Actual_Class,Predicted_Class
0,تشكيله ضمك امام الاتحاد,رياضه,رياضه
1,سبب هذا الاكتشاف الضخم زعزعه وسط العلماء وهذا ...,تكنولوجيا,علوم
2,حذر المركز الأوروبي للوقايه من الأمراض ومكافحت...,صحه,صحه
3,خسوف القلب متي رأيت القلب ترحل عنه حب الله وال...,دين,خواطر واقتباسات
4,تشكيله الفريقين السنغال مصر,رياضه,رياضه
5,الإشتياق لشخص هو نوع من الإدمان لايعاقب عليه ا...,خواطر واقتباسات,خواطر واقتباسات
6,اكتشف العلماء نوعا من السلوك المغناطيسي من شأن...,علوم,علوم
7,التحديث القادم لنظام سيشهد تخلص مايكروسوفت من ...,تكنولوجيا,تكنولوجيا
8,اليوم بدء إدراج وتداول أنابيب الشرق المتكامله ...,اقتصاد,اقتصاد
9,أيها الماضي لا تغيرنا كلما ابتعدنا عنك أيها ال...,خواطر واقتباسات,خواطر واقتباسات


In [ ]:
predicted_class_names = label_encoder.inverse_transform(y_test_pred_int)
actual_class_names = label_encoder.inverse_transform(y_test_int)

# Create a DataFrame to display test tweets with their predicted classes
results_df = pd.DataFrame({
    'Tweet': test_df[TEXT_COL].values,
    'Actual_Class': actual_class_names,
    'Predicted_Class': predicted_class_names
})

results_df.to_csv('predicted_tweets.csv', index=False)
print("DataFrame exported to 'predicted_tweets.csv'")

DataFrame exported to 'predicted_tweets.csv'
